In [ ]:
!pip install pydub
!pip install -U openai-whisper
# !pip install pyrnnoise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ac750e8225bd59e512c3d97cd0d5ede71e708eedc0f021c5d4a15bb403168d8c
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import whisper

model = whisper.load_model("large")

100%|█████████████████████████████████████| 2.88G/2.88G [00:38<00:00, 79.2MiB/s]


### deep learning model for removing noise

In [ ]:
# from pyrnnoise import RNNoise
# import soundfile as sf

# # 1️⃣ Define file paths
# input_audio_path = "1.mp3"
# output_audio_path = "cleaned_audio.wav"

# # 2️⃣ Load audio just to print info (optional)
# audio, sample_rate = sf.read(input_audio_path)
# print(f"Input sample rate: {sample_rate}")

# # 3️⃣ Create the RNNoise denoiser
# denoiser = RNNoise(sample_rate=sample_rate)

# # 4️⃣ Process the audio file and save cleaned output
# for result in denoiser.denoise_wav(input_audio_path, output_audio_path):
#     # result may be an array, so handle its elements
#     # If it’s a NumPy array with one element per channel:
#     if hasattr(result, "__len__"):
#         probs = result
#         # Print each probability
#         print("Frame speech probabilities:", [float(p) for p in probs])
#     else:
#         # If it’s a single scalar (unlikely but safe)
#         print("Speech probability:", float(result))

# print("Denoising completed — saved to:", output_audio_path)


In [ ]:
# !pip install noisereduce

### Spectral Gating / Spectral Subtraction–based


In [ ]:
# import noisereduce as nr
# import librosa

# # load your audio
# audio, sr = librosa.load("1.mp3", sr=None)

# # estimate noise from the first 0.5 seconds
# noise_clip = audio[0:int(0.2*sr)]

# # apply spectral noise reduction
# cleaned = nr.reduce_noise(
#     y=audio,
#     y_noise=noise_clip,
#     n_fft=2048,
#     prop_decrease=1.0,
#     sr=sr
# )

# # save or playback
# import soundfile as sf
# sf.write("cleaned_voice.wav", cleaned, sr)


| Stage                     | Algorithm / Technique                                        |
| ------------------------- | ------------------------------------------------------------ |
| High‑Pass Filter          | **Butterworth IIR** high‑pass filter                         |
| Notch Filter              | **IIR notch filter** at 50 Hz                                |
| Noise Reduction           | **Spectral Gating** (frequency‑domain masking) ([GitHub][1]) |
| Spectrogram Visualization | Standard time–frequency FFT                                  |

[1]: https://github.com/timsainb/noisereduce?utm_source=chatgpt.com "GitHub - timsainb/noisereduce: Noise reduction in python using spectral gating (speech, bioacoustics, audio, time-domain signals)"


In [ ]:
import librosa
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt, iirnotch
import noisereduce as nr
import matplotlib.pyplot as plt

# load the audio
audio_path = "1.mp3"   # صوت التلاوة
y, sr = librosa.load(audio_path, sr=None)
print("Sample Rate:", sr)

# high pass filter
def highpass_filter(signal, sr, cutoff=80):
    nyquist = 0.5 * sr
    normal_cutoff = cutoff / nyquist
    b, a = butter(4, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)
y = highpass_filter(y, sr, cutoff=80)

# notch filter
def notch_filter(signal, sr, freq=50, Q=30):
    nyquist = 0.5 * sr
    w0 = freq / nyquist
    b, a = iirnotch(w0, Q)
    return filtfilt(b, a, signal)
y = notch_filter(y, sr, freq=50)

# noise reduction
y_denoised = nr.reduce_noise(
    y=y,
    sr=sr,
    prop_decrease=0.8,   # لا ترفعها كثير
    stationary=False
)

# save the result
sf.write(f"{audio_path.split('.')[0]}_cleaned.wav", y_denoised, sr)
print("تم حفظ الصوت المنظف ✔")


# # spectrum dislpay before and after filtering
# plt.figure(figsize=(12,4))
# plt.specgram(y, Fs=sr)
# plt.title("Before Filtering")
# plt.show()

# plt.figure(figsize=(12,4))
# plt.specgram(y_denoised, Fs=sr)
# plt.title("After Filtering")
# plt.show()

Sample Rate: 22050
تم حفظ الصوت المنظف ✔


available models = ['tiny.en', 'tiny', 'base.en', 'base', 'small.en', 'small', 'medium.en', 'medium', 'large-v1', 'large-v2', 'large-v3', 'large', 'large-v3-turbo', 'turbo']

In [ ]:
import whisper

model = whisper.load_model("medium")

audio_path_for_wisper= "1_cleaned.wav"

result = model.transcribe(
    audio_path_for_wisper,
    language="ar",
     word_timestamps=True,
)
print(result)
print(result["text"])


{'text': ' بسم الله الرحمن الرحيم الحمد لله رب العالمين الرحمن الرحيم مالك يوم الدين إياك نعبد وإياك نستعين إهدن الصراط المستقيم صراط الذين أنعمت عليهم غير المغضوب عليهم ولا الضالين', 'segments': [{'id': 0, 'seek': 0, 'start': np.float64(0.0), 'end': 4.64, 'text': ' بسم الله الرحمن الرحيم', 'tokens': [50364, 4724, 38251, 21984, 34892, 5016, 27842, 34892, 5016, 32640, 50596], 'temperature': 0.0, 'avg_logprob': -0.30743477262299634, 'compression_ratio': 1.8526315789473684, 'no_speech_prob': 0.01796429604291916, 'words': [{'word': ' بسم', 'start': np.float64(0.0), 'end': np.float64(0.54), 'probability': np.float64(0.7766459882259369)}, {'word': ' الله', 'start': np.float64(0.54), 'end': np.float64(1.28), 'probability': np.float64(0.978307843208313)}, {'word': ' الرحمن', 'start': np.float64(1.28), 'end': np.float64(2.38), 'probability': np.float64(0.9780131975809733)}, {'word': ' الرحيم', 'start': np.float64(2.38), 'end': 4.64, 'probability': np.float64(0.9971399108568827)}]}, {'id': 1, 's

In [ ]:
timestamps = []
for segment in result["segments"][3]["words"]:
  print("(",segment["start"],",", segment["end"], ")")
  print(segment["word"])
  timestamps.append((segment["start"], segment["end"]))

( 16.1 , 17.22 )
 مالك
( 17.22 , 17.84 )
 يوم
( 17.84 , 19.92 )
 الدين


In [ ]:
# split voices
from pydub import AudioSegment

# Load audio file
audio = AudioSegment.from_file(audio_path_for_wisper, format="mp3")

for i, (start, end) in enumerate(timestamps, start=1):
    clip = audio[int(start * 1000):int(end * 1000)]
    clip.export(f"clip_{i}.mp3", format="mp3")

print("Done splitting audio")


Done splitting audio
